# Part II · Lesson 3 — Draft an office email reply

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

### Import the model and messages

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

### Connect to our existing Groq model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

### Describe the email record

In [ ]:
class EmailState(TypedDict):
    email: str
    tone: str
    draft: str
    status: str

### Make the model call inside a node

In [ ]:
def draft_reply(state: EmailState) -> dict:
    reply = model.invoke([
        SystemMessage(
            f"Draft a {state['tone']} office email reply. "
            "Acknowledge the request and ask for a preferred time. "
            "Do not confirm calendar availability."
        ),
        HumanMessage(state["email"]),
    ])
    return {"draft": reply.text, "status": "draft ready"}

### Connect the one-node graph

In [ ]:
builder = StateGraph(EmailState)
builder.add_node("draft_reply", draft_reply)
builder.add_edge(START, "draft_reply")
builder.add_edge("draft_reply", END)
graph = builder.compile()

### Request and inspect the draft

In [ ]:
email_input = {
    "email": "Hi, can we move Thursday's project review to Friday afternoon?",
    "tone": "friendly and concise", "draft": "", "status": "new",
}
response = graph.invoke(email_input)
print(response["draft"])
print(response["status"])